In [1]:
import os
os.environ["CALITP_BQ_MAX_BYTES"] = str(800_000_000_000)

import shared_utils
import pandas as pd
import geopandas as gpd

import gcsfs
from calitp_data_analysis import get_fs
from calitp_data_analysis import geography_utils, utils
fs = get_fs()
import re
import google.auth
import os
import gcsfs
credentials, project = google.auth.default()
fs = gcsfs.GCSFileSystem()

In [2]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships/AHSC_2026'

In [3]:
# url_tot = "https://lehd.ces.census.gov/data/lodes/LODES8/ca/wac/ca_wac_S000_JT00_2023.csv.gz"
# url_prv = "https://lehd.ces.census.gov/data/lodes/LODES8/ca/wac/ca_wac_S000_JT01_2023.csv.gz"


In [4]:
# Workplace Area Characteristics (WAC)
url_wac_tot = "https://lehd.ces.census.gov/data/lodes/LODES8/ca/wac/ca_wac_S000_JT00_2023.csv.gz"
url_wac_prv = "https://lehd.ces.census.gov/data/lodes/LODES8/ca/wac/ca_wac_S000_JT01_2023.csv.gz"

wac_tot = pd.read_csv(url_wac_tot, compression="gzip")
wac_prv = pd.read_csv(url_wac_prv, compression="gzip")

# Residence Area Characteristics (RAC)
url_rac_tot = "https://lehd.ces.census.gov/data/lodes/LODES8/ca/rac/ca_rac_S000_JT00_2023.csv.gz"
url_rac_prv = "https://lehd.ces.census.gov/data/lodes/LODES8/ca/rac/ca_rac_S000_JT01_2023.csv.gz"

rac_tot = pd.read_csv(url_rac_tot, compression="gzip")
rac_prv = pd.read_csv(url_rac_prv, compression="gzip")

In [5]:
# lodes_tot = pd.read_csv(url_tot, compression='gzip')
# lodes_prv = pd.read_csv(url_prv, compression='gzip')

In [6]:
wac_tot.head(5)

,w_geocode,C000,CA01,CA02,CA03,CE01,CE02,CE03,CNS01,CNS02,...,CFA02,CFA03,CFA04,CFA05,CFS01,CFS02,CFS03,CFS04,CFS05,createdate
0,60014001001003,21,2,10,9,0,0,21,0,0,...,0,0,0,0,0,0,0,0,0,20251202
1,60014001001010,3,0,1,2,0,0,3,0,0,...,0,0,0,0,0,0,0,0,0,20251202
2,60014001001011,1,0,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,20251202
3,60014001001013,3,0,2,1,0,0,3,0,0,...,0,0,0,0,0,0,0,0,0,20251202
4,60014001001015,3,0,1,2,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,20251202


In [7]:


# for df in [lodes_tot, lodes_prv]:
#     df['GEOID'] = df['w_geocode'].astype(str).str.zfill(15).str[:12]

# jobs_bg = lodes_tot.groupby('GEOID').agg(jobs_tot=('C000', 'sum')).reset_index()
# jobs_prv = lodes_prv.groupby('GEOID').agg(jobs_prv=('C000', 'sum')).reset_index()

# jobs_bg = jobs_bg.merge(jobs_prv, on='GEOID', how='left')
# jobs_bg.rename(columns={'GEOID': 'h_geocode'}, inplace=True)
# jobs_bg['segment'] = 'S000'
# jobs_bg['year'] = 2023
# jobs_bg = jobs_bg[['h_geocode', 'segment', 'year', 'jobs_tot', 'jobs_prv']]

In [8]:
datasets = {
    "wac": (wac_tot, wac_prv, "w_geocode"),
    "rac": (rac_tot, rac_prv, "h_geocode"),
}

results = {}

for name, (df_tot, df_prv, geocode_col) in datasets.items():
    # Create 12-digit block group GEOID
    for df in [df_tot, df_prv]:
        df["GEOID"] = df[geocode_col].astype(str).str.zfill(15).str[:12]

    bg_tot = (
        df_tot.groupby("GEOID", as_index=False)
              .agg(jobs_tot=("C000", "sum"))
    )

    bg_prv = (
        df_prv.groupby("GEOID", as_index=False)
              .agg(jobs_prv=("C000", "sum"))
    )

    bg = (
        bg_tot.merge(bg_prv, on="GEOID", how="left")
              .rename(columns={"GEOID": geocode_col})
    )

    bg["segment"] = "S000"
    bg["year"] = 2023

    results[name] = bg[
        [geocode_col, "segment", "year", "jobs_tot", "jobs_prv"]
    ]

# Outputs
wac_bg = results["wac"]   # Jobs by workplace block group
rac_bg = results["rac"]   # Workers by residence block group

In [9]:
# #Sort by GEOID
# jobs_bg = jobs_bg.sort_values(by='h_geocode')

# #Create jobs_fed = jobs_total - jobs_prv
# jobs_bg['jobs_fed'] = jobs_bg['jobs_tot'] - jobs_bg['jobs_prv']


In [10]:
for df, geocode_col in [(wac_bg, "w_geocode"), (rac_bg, "h_geocode")]:
    df.sort_values(by=geocode_col, inplace=True)
    df["jobs_fed"] = df["jobs_tot"] - df["jobs_prv"]

In [11]:
# jobs_bg['GEOID'] = jobs_bg['h_geocode'].astype(str).str[:12]

In [12]:
rac_bg["GEOID"] = rac_bg["h_geocode"].astype(str).str[:12]
wac_bg["GEOID"] = wac_bg["w_geocode"].astype(str).str[:12]

In [13]:
# grouped_df = jobs_bg.groupby('GEOID', as_index=False).agg({
#     'jobs_tot': 'sum',
#     'jobs_prv': 'sum',
#     'jobs_fed': 'sum',
#     'year': 'first'
# })

In [14]:
wac_bg_grouped = (
    wac_bg.groupby("GEOID", as_index=False)
          .agg(
              jobs_tot_work=("jobs_tot", "sum"),
              jobs_prv_work=("jobs_prv", "sum"),
              jobs_fed_work=("jobs_fed", "sum"),
          )
)

rac_bg_grouped = (
    rac_bg.groupby("GEOID", as_index=False)
          .agg(
              jobs_tot_home=("jobs_tot", "sum"),
              jobs_prv_home=("jobs_prv", "sum"),
              jobs_fed_home=("jobs_fed", "sum"),
          )
)

jobs_bg = wac_bg_grouped.merge(rac_bg_grouped, on="GEOID", how="outer")
jobs_bg["year"] = 2023

In [17]:
jobs_bg.head(5)

,GEOID,jobs_tot_work,jobs_prv_work,jobs_fed_work,jobs_tot_home,jobs_prv_home,jobs_fed_home,year
0,060014001001,136.0,131.0,5.0,837.0,776.0,61.0,2023
1,060014001002,145.0,133.0,12.0,599.0,555.0,44.0,2023
2,060014002001,683.0,617.0,66.0,467.0,433.0,34.0,2023
3,060014002002,420.0,385.0,35.0,500.0,459.0,41.0,2023
4,060014003001,320.0,242.0,78.0,497.0,460.0,37.0,2023


In [15]:
def export_gdf(gdf, filename: str):
    
    gdf.to_parquet(f"{filename}.parquet")
    
    fs.put(
        f"{filename}.parquet",
        f"{GCS_FILE_PATH}/{filename}.parquet",
        token = credentials.token
    )
    
    os.remove(f"{filename}.parquet")
    print(f"saved {GCS_FILE_PATH}/{filename}.parquet")
    
    return

In [16]:
export_gdf(jobs_bg, "job_density_blockwithrac_2023")

saved gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships/AHSC_2026/job_density_blockwithrac_2023.parquet
